In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [16]:
from statsmodels.tsa.api import ExponentialSmoothing, SimpleExpSmoothing

In [2]:
# Load test data
df_test = pd.read_csv('assignment_data_test.csv')
df_test.head()

,Timestamp,year,month,day,hour
0,2019-01-01 00:00:00,2019,1,1,0
1,2019-01-01 01:00:00,2019,1,1,1
2,2019-01-01 02:00:00,2019,1,1,2
3,2019-01-01 03:00:00,2019,1,1,3
4,2019-01-01 04:00:00,2019,1,1,4


In [9]:
# List data types of test data
df_test.dtypes

Timestamp    datetime64[ns]
year                  int64
month                 int64
day                   int64
hour                  int64
dtype: object

In [ ]:
# Convert Timestamp column to datetime
df_test['Timestamp'] = pd.to_datetime(df_test.Timestamp)

In [8]:
# List data types of test data
df_test.dtypes

Timestamp    datetime64[ns]
year                  int64
month                 int64
day                   int64
hour                  int64
dtype: object

In [12]:
# Load train data 
df_train = pd.read_csv('assignment_data_train.csv')
df_train.head()

,Timestamp,year,month,day,hour,trips
0,2018-01-01 00:00:00,2018,1,1,0,16714
1,2018-01-01 01:00:00,2018,1,1,1,19041
2,2018-01-01 02:00:00,2018,1,1,2,16590
3,2018-01-01 03:00:00,2018,1,1,3,12626
4,2018-01-01 04:00:00,2018,1,1,4,8739


In [ ]:
# List data types of train data
df_train.dtypes

In [ ]:
# Convert Timestamp column to datetime 
df_train['Timestamp'] = pd.to_datetime(df_train.Timestamp)

In [18]:
trips = df_train['trips']
trips.index = df_train['Timestamp']
trips.index.freq = trips.index.inferred_freq

alpha020 = SimpleExpSmoothing(trips).fit(
                                        smoothing_level=0.2,
                                        optimized=False)

alpha050 = SimpleExpSmoothing(trips).fit(
                                        smoothing_level=0.5,
                                        optimized=False)

alpha080 = SimpleExpSmoothing(trips).fit(
                                        smoothing_level=0.8,
                                        optimized=False)

forecast020 = alpha020.forecast(3)
forecast050 = alpha050.forecast(3)
forecast080 = alpha080.forecast(3)

In [20]:
import plotly.graph_objects as go

In [22]:
# Plotting our data

smoothData = pd.DataFrame([trips.values, alpha020.fittedvalues.values,  alpha050.fittedvalues.values,  alpha080.fittedvalues.values]).T
smoothData.columns = ['Truth', 'alpha=0.2', 'alpha=0.5', 'alpha=0.8']
smoothData.index = trips.index


In [27]:
smoothData.head()

,Truth,alpha=0.2,alpha=0.5,alpha=0.8
Timestamp,,,,
2018-01-01 00:00:00,16714.0,16714.000,16714.000,16714.000
2018-01-01 01:00:00,19041.0,16714.000,16714.000,16714.000
2018-01-01 02:00:00,16590.0,17179.400,17877.500,18575.600
2018-01-01 03:00:00,12626.0,17061.520,17233.750,16987.120
2018-01-01 04:00:00,8739.0,16174.416,14929.875,13498.224


In [36]:
fig = px.line(smoothData, y = ['Truth', 'alpha=0.2', 'alpha=0.5', 'alpha=0.8'], 
        x = smoothData.index,
        color_discrete_map={"Truth": 'blue',
                           'alpha=0.2': 'red', 
                            'alpha=0.5':'green', 
                            'alpha=0.8':'purple'}
       )

fig.update_xaxes(range=[smoothData.index[-50], forecast020.index[-1]])
fig.update_yaxes(range=[0, 24000])

# Incorporating the Forecasts
fig.add_trace(go.Scatter(x=forecast020.index, y = forecast020.values, name='Forecast alpha=0.2', line={'color':'red'}))
fig.add_trace(go.Scatter(x=forecast050.index, y = forecast050.values, name='Forecast alpha=0.5', line={'color':'green'}))
fig.add_trace(go.Scatter(x=forecast080.index, y = forecast080.values, name='Forecast alpha=0.8', line={'color':'purple'}))

In [63]:
# Streamlined Modeling
alphaBest = SimpleExpSmoothing(trips).fit()

/Users/josephcoldanghise/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/holtwinters/model.py:915: ConvergenceWarning:

Optimization failed to converge. Check mle_retvals.



In [64]:
alphaBest2 = SimpleExpSmoothing(trips).fit(optimized=True)

/Users/josephcoldanghise/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/holtwinters/model.py:915: ConvergenceWarning:

Optimization failed to converge. Check mle_retvals.



In [48]:
forecast = alphaBest.forecast(3)

In [49]:
smoothData = pd.DataFrame([trips.values, alphaBest.fittedvalues.values]).T
smoothData.columns = ['Truth', 'Best Fit Model']
smoothData.index = trips.index

fig = px.line(smoothData, y = ['Truth', 'Best Fit Model'], 
        x = smoothData.index,
        color_discrete_map={"Truth": 'blue',
                           'Best Fit Model': 'red'}
       )

fig.update_xaxes(range=[smoothData.index[-50], forecast.index[-1]])
fig.update_yaxes(range=[0, 24000])

# Incorporating the Forecasts

fig.add_trace(go.Scatter(x=forecast.index, y = forecast.values, name='Forecast', line={'color':'red'}))

In [65]:
# Linear trend
trend = ExponentialSmoothing(trips, trend='add').fit(optimized=True)
# Linear trend with damping
dampedTrend = ExponentialSmoothing(trips, trend='add', damped_trend=True).fit(use_brute=True)

forecast_t = trend.forecast(10)
forecast_dt = dampedTrend.forecast(10)

/Users/josephcoldanghise/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/holtwinters/model.py:915: ConvergenceWarning:

Optimization failed to converge. Check mle_retvals.

/Users/josephcoldanghise/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/holtwinters/model.py:915: ConvergenceWarning:

Optimization failed to converge. Check mle_retvals.



In [66]:

# Plotting our data

smoothData = pd.DataFrame([trips.values, trend.fittedvalues.values, dampedTrend.fittedvalues.values]).T
smoothData.columns = ['Truth', 'Trend', 'Damped Trend']
smoothData.index = trips.index

fig = px.line(smoothData, y = ['Truth', 'Trend', 'Damped Trend'], 
        x = smoothData.index,
        color_discrete_map={"Truth": 'blue',
                           'Trend': 'red',
                            'Damped Trend': 'green'
                           },
              title='Linear and Damped Trends'
       )

fig.update_xaxes(range=[smoothData.index[-50], forecast_t.index[-1]])
fig.update_yaxes(range=[0, 24000])


# Incorporating the Forecasts

fig.add_trace(go.Scatter(x=forecast_t.index, y = forecast_t.values, name='Forecast Trend', line={'color':'red'}))
fig.add_trace(go.Scatter(x=forecast_dt.index, y = forecast_dt.values, name='Forecast Damped Trend', line={'color':'green'}))

In [57]:
# Linear trend
trend = ExponentialSmoothing(trips, trend='add', seasonal='add').fit()
# Linear trend with damping
dampedTrend = ExponentialSmoothing(trips, trend='mul', seasonal='add', damped_trend=True).fit(use_brute=True)

forecast_t = trend.forecast(10)
forecast_dt = dampedTrend.forecast(10)

/Users/josephcoldanghise/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/holtwinters/model.py:915: ConvergenceWarning:

Optimization failed to converge. Check mle_retvals.

/Users/josephcoldanghise/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/holtwinters/model.py:83: RuntimeWarning:

overflow encountered in matmul

/Users/josephcoldanghise/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/holtwinters/model.py:915: ConvergenceWarning:

Optimization failed to converge. Check mle_retvals.



In [58]:
smoothData = pd.DataFrame([trips.values, trend.fittedvalues.values, dampedTrend.fittedvalues.values]).T
smoothData.columns = ['Truth', 'Trend', 'Damped Trend']
smoothData.index = trips.index

fig = px.line(smoothData, y = ['Truth', 'Trend', 'Damped Trend'], 
        x = smoothData.index,
        color_discrete_map={"Truth": 'blue',
                           'Trend': 'red',
                            'Damped Trend': 'green'
                           },
              title='With Seasonality'
       )

fig.update_xaxes(range=[smoothData.index[-50], forecast_t.index[-1]])
fig.update_yaxes(range=[0, 24000])


# Incorporating the Forecasts

fig.add_trace(go.Scatter(x=forecast_t.index, y = forecast_t.values, name='Forecast Trend', line={'color':'red'}))
fig.add_trace(go.Scatter(x=forecast_dt.index, y = forecast_dt.values, name='Forecast Damped Trend', line={'color':'green'}))

In [88]:
# Model test

model = ExponentialSmoothing(trips, trend='add')

In [89]:
fit = model.fit(optimized=True)

/Users/josephcoldanghise/opt/anaconda3/lib/python3.9/site-packages/statsmodels/tsa/holtwinters/model.py:915: ConvergenceWarning:

Optimization failed to converge. Check mle_retvals.



In [90]:
fitted = fit.fittedvalues            # same index as y_train

In [91]:
fitted

Timestamp
2018-01-01 00:00:00    17837.672727
2018-01-01 01:00:00    13718.026999
2018-01-01 02:00:00    21282.670116
2018-01-01 03:00:00    14235.887600
2018-01-01 04:00:00     8662.644716
                           ...     
2018-12-31 19:00:00    15455.515831
2018-12-31 20:00:00    15502.475537
2018-12-31 21:00:00    14005.102842
2018-12-31 22:00:00    13790.996937
2018-12-31 23:00:00     7388.407890
Freq: H, Length: 8760, dtype: float64

In [92]:
residuals = trips - fitted

In [93]:
residuals

Timestamp
2018-01-01 00:00:00   -1123.672727
2018-01-01 01:00:00    5322.973001
2018-01-01 02:00:00   -4692.670116
2018-01-01 03:00:00   -1609.887600
2018-01-01 04:00:00      76.355284
                          ...     
2018-12-31 19:00:00    -579.515831
2018-12-31 20:00:00   -1068.475537
2018-12-31 21:00:00     110.897158
2018-12-31 22:00:00   -3061.996937
2018-12-31 23:00:00    1210.592110
Freq: H, Length: 8760, dtype: float64

In [94]:
n_forecast = 12
forecast = fit.forecast(n_forecast) 

In [107]:
df_test.head()

,Timestamp,year,month,day,hour
0,2019-01-01 00:00:00,2019,1,1,0
1,2019-01-01 01:00:00,2019,1,1,1
2,2019-01-01 02:00:00,2019,1,1,2
3,2019-01-01 03:00:00,2019,1,1,3
4,2019-01-01 04:00:00,2019,1,1,4


In [115]:
df_test.index = df_test['Timestamp']
df_test.index.freq  = df_test.index.inferred_freq

In [ ]:
y_test  = df_test.iloc[-h:]

In [119]:
df_test.index

DatetimeIndex(['2019-01-01 00:00:00', '2019-01-01 01:00:00',
               '2019-01-01 02:00:00', '2019-01-01 03:00:00',
               '2019-01-01 04:00:00', '2019-01-01 05:00:00',
               '2019-01-01 06:00:00', '2019-01-01 07:00:00',
               '2019-01-01 08:00:00', '2019-01-01 09:00:00',
               ...
               '2019-01-31 14:00:00', '2019-01-31 15:00:00',
               '2019-01-31 16:00:00', '2019-01-31 17:00:00',
               '2019-01-31 18:00:00', '2019-01-31 19:00:00',
               '2019-01-31 20:00:00', '2019-01-31 21:00:00',
               '2019-01-31 22:00:00', '2019-01-31 23:00:00'],
              dtype='datetime64[ns]', name='Timestamp', length=744, freq='H')

In [120]:
trips.index

DatetimeIndex(['2018-01-01 00:00:00', '2018-01-01 01:00:00',
               '2018-01-01 02:00:00', '2018-01-01 03:00:00',
               '2018-01-01 04:00:00', '2018-01-01 05:00:00',
               '2018-01-01 06:00:00', '2018-01-01 07:00:00',
               '2018-01-01 08:00:00', '2018-01-01 09:00:00',
               ...
               '2018-12-31 14:00:00', '2018-12-31 15:00:00',
               '2018-12-31 16:00:00', '2018-12-31 17:00:00',
               '2018-12-31 18:00:00', '2018-12-31 19:00:00',
               '2018-12-31 20:00:00', '2018-12-31 21:00:00',
               '2018-12-31 22:00:00', '2018-12-31 23:00:00'],
              dtype='datetime64[ns]', name='Timestamp', length=8760, freq='H')

In [122]:
y_pred = fit.predict(start=df_test.index[0], end=df_test.index[-1])

In [95]:
pred_holdout = fit.predict(start=trips.index[0], end=trips.index[-1])

In [96]:
mape = (np.abs((trips - pred_holdout) / trips)).mean() * 100
print(f"MAPE on holdout: {mape:.2f}%")

MAPE on holdout: 88.70%


In [97]:
e = trips - pred_holdout

In [98]:
rmse = np.sqrt(np.mean(e**2))

In [99]:
rmse

1945.983782590574